# Lab 03 - Evaluations (Build -> Evaluations)

**Portal location:** _Microsoft Foundry -> Build -> Evaluations_

Before Zava rolls `zava-support-bot` to 10% of live traffic, the platform team needs evidence that it:

1. **Answers accurately** using groundedness, relevance, coherence, and fluency.
2. **Responds safely** across violence, hate/unfairness, self-harm, and sexual-content criteria.
3. **Handles agent tasks correctly** using intent resolution, task adherence, and tool-call accuracy.

This version uses Azure AI Projects SDK v2 and the project-scoped OpenAI evaluation API. It creates three cloud evaluations: quality and safety runs over a versioned dataset, plus an agent run over one inline tool-calling trace. Definitions, runs, and results are stored in the Foundry project and appear in the new portal.

> **Authentication note:** Sign in to the tenant in `AZURE_TENANT_ID` before running the notebook. Device-code login works when automatic browser launch is unavailable: `az login --tenant <tenant-id> --use-device-code`.

> References:
> - [Run evaluations from the Microsoft Foundry portal](https://learn.microsoft.com/en-us/azure/foundry/how-to/evaluate-generative-ai-app)
> - [Cloud evaluation with the Microsoft Foundry SDK](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/cloud-evaluation)

In [ ]:
# What this cell does: load configuration, authenticate to the correct tenant,

# and create clients for both Foundry project operations and OpenAI evaluation APIs.

import os, json, time

from pathlib import Path

from pprint import pprint



from dotenv import load_dotenv

from azure.identity import AzureCliCredential

from azure.ai.projects import AIProjectClient

from azure.ai.projects.models import TestingCriterionAzureAIEvaluator

from openai.types.eval_create_params import DataSourceConfigCustom



# Load the project endpoint, model deployments, and tenant from the repository .env.

# override=True prevents stale shell variables from selecting a different tenant.

load_dotenv(Path.cwd().parent / ".env", override=True)



# Reuse the Azure CLI login, explicitly scoped to the tenant that owns the project.

credential = AzureCliCredential(tenant_id=os.environ["AZURE_TENANT_ID"])

PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]

MODEL = os.environ["FOUNDRY_MODEL_NAME"]



# The judge can use a dedicated deployment; otherwise it reuses the application model.

JUDGE_MODEL = os.getenv("FOUNDRY_EVALUATION_MODEL_NAME", MODEL)



# The project client manages datasets; its OpenAI client creates and runs evaluations.

project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

openai_client = project.get_openai_client()

## 1. Generate candidate responses

This section loads eight hand-authored test cases and the Zava return policy. It calls the deployed application model once per query, then writes `query`, generated `response`, evaluator `context`, and `ground_truth` to a local JSONL file.

> Running the code cell invokes the deployed model and overwrites `data/zava_support_bot_responses.jsonl` with fresh responses.

In [ ]:
# What this cell does: run the support bot against eight test questions and

# build a JSONL dataset containing every input required by the evaluators.



# Load hand-authored questions, grounding context, and expected answers.

eval_path = Path("data/zava_eval_dataset.jsonl")

records = [

    json.loads(line)

    for line in eval_path.read_text(encoding="utf-8").splitlines()

    if line.strip()

]

policy = Path("data/zava_return_policy.md").read_text(encoding="utf-8")

print(f"Loaded {len(records)} eval records")



# Ground the application model in the same return policy for every test case.

SYSTEM = (

    "You are the Zava support bot. Answer ONLY using the return policy below. "

    "If the policy doesn't cover it, say you don't know.\n\n"

    f"<policy>\n{policy}\n</policy>"

)



# Generate one candidate answer per query through the project-scoped OpenAI client.

# Keep context and ground truth beside the response for later evaluator mappings.

responses = []

for record in records:

    reply = openai_client.chat.completions.create(

        model=MODEL,

        messages=[

            {"role": "system", "content": SYSTEM},

            {"role": "user", "content": record["query"]},

        ],

        max_completion_tokens=250,

    ).choices[0].message.content

    responses.append({

        "query": record["query"],

        "response": reply,

        "context": record["context"],

        "ground_truth": record["ground_truth"],

    })



# Write one JSON object per line, which the Foundry dataset upload API accepts.

eval_out = Path("data/zava_support_bot_responses.jsonl")

eval_out.write_text(

    "\n".join(json.dumps(response) for response in responses),

    encoding="utf-8",

)

print(f"Wrote {eval_out} ({len(responses)} rows)")

## 2. Define quality evaluators

The cloud service validates each JSONL row against an item schema, then maps its fields to four Microsoft-curated evaluators. AI-assisted evaluators use `JUDGE_MODEL`; their output includes a natural-scale score, pass/fail label, threshold, and reason.

- **Groundedness:** checks whether claims are supported by `context`.
- **Relevance:** checks whether `response` addresses `query`.
- **Coherence:** checks logical structure and readability.
- **Fluency:** checks grammar and language quality.

In [ ]:
# What this cell does: define the dataset contract and configure four cloud
# quality evaluators. It does not upload data or start an evaluation run.

# Tell Foundry which fields every JSONL item contains and which are mandatory.
data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "response": {"type": "string"},
            "context": {"type": "string"},
            "ground_truth": {"type": "string"},
        },
        "required": ["query", "response", "context", "ground_truth"],
    },
)

# Map each evaluator's expected inputs to fields in the uploaded dataset item.
# The double braces are Foundry template syntax, not Python interpolation.
quality_criteria = [
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name=name,
        evaluator_name=f"builtin.{name}",
        initialization_parameters={"model": JUDGE_MODEL},
        data_mapping={field: f"{{{{item.{field}}}}}" for field in fields},
    )
    for name, fields in {
        "groundedness": ("query", "response", "context"),
        "relevance": ("query", "response"),
        "coherence": ("query", "response"),
        "fluency": ("response",),
    }.items()
]

# Preview the resolved evaluator catalog names before creating a paid cloud run.
print(f"Configured {len(quality_criteria)} cloud quality evaluators (judge: {JUDGE_MODEL})")
pprint([criterion["evaluator_name"] for criterion in quality_criteria])

## 3. Upload data and run the quality evaluation

A Foundry cloud evaluation has separate resources:

1. A **dataset version** stores the generated JSONL in the project.
2. An **evaluation definition** stores the item schema and testing criteria.
3. An **evaluation run** applies that definition to a selected data source.

The timestamp makes each dataset version and run name unique. Rerunning the submission cell creates another dataset version, evaluation definition, and billable cloud run; it does not update the previous run.

In [ ]:
# What this cell does: upload the response file, create a quality evaluation
# definition, and queue one cloud run. Rerunning it creates new project assets.

# Use a sortable timestamp so every upload becomes a distinct dataset version.
dataset_version = time.strftime("%Y%m%d%H%M%S")
dataset = project.datasets.upload_file(
    name="zava-support-bot-responses",
    version=dataset_version,
    file_path=str(eval_out),
)
print(f"Uploaded dataset version {dataset_version}: {dataset.id}")

# Store the schema and quality criteria as a reusable project evaluation definition.
quality_eval = openai_client.evals.create(
    name="zava-support-bot-quality",
    data_source_config=data_source_config,
    testing_criteria=quality_criteria,
)

# Bind this run to the uploaded JSONL dataset by its Foundry file ID.
quality_run = openai_client.evals.runs.create(
    eval_id=quality_eval.id,
    name=f"zava-quality-{dataset_version}",
    data_source={
        "type": "jsonl",
        "source": {"type": "file_id", "id": dataset.id},
    },
)
print(f"Started quality evaluation: {quality_eval.id}")
print(f"Run: {quality_run.id} ({quality_run.status})")

In [ ]:
# What this cell does: wait for the asynchronous quality run, print its new
# Foundry report URL, and retrieve the per-item results through the SDK.

# Refresh the run every five seconds until it reaches a terminal state.
while True:
    quality_run = openai_client.evals.runs.retrieve(
        eval_id=quality_eval.id,
        run_id=quality_run.id,
    )
    if quality_run.status in ("completed", "failed", "canceled"):
        break
    print(f"Quality run status: {quality_run.status}")
    time.sleep(5)

# report_url opens this exact run in the new Foundry portal.
print(f"Quality run finished: {quality_run.status}")
print("Open in new Foundry:", quality_run.report_url)

# Output items contain row-level evaluator scores, labels, thresholds, and reasons.
quality_output_items = list(
    openai_client.evals.runs.output_items.list(
        eval_id=quality_eval.id,
        run_id=quality_run.id,
    )
)
print(f"Scored rows: {len(quality_output_items)}")

## 4. Run risk and safety evaluation

The same uploaded query/response dataset is reused for a separate evaluation definition containing four project-hosted safety criteria. Keeping quality and safety in separate runs makes their status, report, and failure analysis easier to review independently.

> Safety output item counts can be lower than dataset row counts when a row or evaluator produces no scorable result. Open the report URL to inspect row-level status and reasons.

In [ ]:
# What this cell does: configure four safety evaluators for the existing
# query/response schema. It only builds configuration; it starts no cloud run.

# Every safety criterion consumes the same query and generated response fields.
safety_criteria = [
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name=alias,
        evaluator_name=f"builtin.{evaluator_name}",
        initialization_parameters={"model": JUDGE_MODEL},
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{item.response}}",
        },
    )
    for alias, evaluator_name in {
        "violence": "violence",
        "hate_unfairness": "hate_unfairness",
        "self_harm": "self_harm",
        "sexual": "sexual",
    }.items()
]

# Preview catalog names before the following cell creates a billable cloud run.
print(f"Configured {len(safety_criteria)} cloud safety evaluators")
pprint([criterion["evaluator_name"] for criterion in safety_criteria])

In [ ]:
# What this cell does: create, run, poll, and retrieve a separate safety
# evaluation using the dataset version uploaded by the quality workflow.

# Create a safety-specific definition while reusing the same item schema.
safety_eval = openai_client.evals.create(
    name="zava-support-bot-safety",
    data_source_config=data_source_config,
    testing_criteria=safety_criteria,
)

# Reuse the existing project dataset instead of uploading duplicate input data.
safety_run = openai_client.evals.runs.create(
    eval_id=safety_eval.id,
    name=f"zava-safety-{dataset_version}",
    data_source={
        "type": "jsonl",
        "source": {"type": "file_id", "id": dataset.id},
    },
)
print(f"Started safety evaluation: {safety_eval.id}")

# Safety evaluation is asynchronous, just like the quality run.
while True:
    safety_run = openai_client.evals.runs.retrieve(
        eval_id=safety_eval.id,
        run_id=safety_run.id,
    )
    if safety_run.status in ("completed", "failed", "canceled"):
        break
    print(f"Safety run status: {safety_run.status}")
    time.sleep(5)

# Print the deep link and retrieve all scorable row-level outputs.
print(f"Safety run finished: {safety_run.status}")
print("Open in new Foundry:", safety_run.report_url)
safety_output_items = list(
    openai_client.evals.runs.output_items.list(
        eval_id=safety_eval.id,
        run_id=safety_run.id,
    )
)
print(f"Scored rows: {len(safety_output_items)}")

## 5. Evaluate an agent trace

Agent evaluators need structured interaction data rather than only plain query/response strings. This example contains a user request, assistant tool call, tool result, final answer, and the tool definition used to validate the call.

- `builtin.intent_resolution` checks whether the request was understood and resolved.
- `builtin.task_adherence` checks whether behavior stayed aligned with the task.
- `builtin.tool_call_accuracy` checks tool choice and arguments.

Unlike the quality and safety examples, this one-row trace is sent with `file_content`; it is not uploaded as a reusable dataset version. Running the cell creates a separate cloud evaluation and run named for `zava-order-tracker-agent`.

In [ ]:
# What this cell does: package one tool-calling interaction, configure agent
# evaluators, submit the trace inline, and retrieve the completed cloud results.

# Model the full interaction: user message, tool call, tool result, and final answer.
trace = {
    "query": [
        {
            "role": "user",
            "content": [{"type": "text", "text": "Where's my order Z12345?"}],
        },
    ],
    "response": [
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": "Let me check order Z12345 for you."},
                {
                    "type": "tool_call",
                    "tool_call_id": "call_order_1",
                    "name": "get_order",
                    "arguments": {"order_id": "Z12345"},
                },
            ],
        },
        {
            "role": "tool",
            "tool_call_id": "call_order_1",
            "content": [
                {
                    "type": "tool_result",
                    "tool_result": {
                        "order": "Z12345",
                        "status": "in transit",
                        "eta": "Fri",
                    },
                },
            ],
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": "Your order Z12345 is in transit and should arrive Friday.",
                },
            ],
        },
    ],
    # Supply the function schema so evaluators can validate tool choice and arguments.
    "tool_definitions": [
        {
            "name": "get_order",
            "description": "Look up shipping status for an order.",
            "parameters": {
                "type": "object",
                "properties": {"order_id": {"type": "string"}},
                "required": ["order_id"],
            },
        },
    ],
}

# Agent traces use arrays of typed messages instead of plain string fields.
agent_data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "array"},
            "response": {"type": "array"},
            "tool_definitions": {"type": "array"},
        },
        "required": ["query", "response", "tool_definitions"],
    },
)

# Give all three agent evaluators the conversation and available tool contract.
agent_criteria = [
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name=name,
        evaluator_name=f"builtin.{name}",
        initialization_parameters={"model": JUDGE_MODEL},
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{item.response}}",
            "tool_definitions": "{{item.tool_definitions}}",
        },
    )
    for name in ("intent_resolution", "task_adherence", "tool_call_accuracy")
]

# Create a dedicated definition for this structured agent-evaluation schema.
agent_eval = openai_client.evals.create(
    name="zava-order-tracker-agent",
    data_source_config=agent_data_source_config,
    testing_criteria=agent_criteria,
)

# Send the single trace inline. file_content avoids registering a one-row dataset.
agent_run = openai_client.evals.runs.create(
    eval_id=agent_eval.id,
    name=f"zava-agent-{dataset_version}",
    data_source={
        "type": "jsonl",
        "source": {
            "type": "file_content",
            "content": [{"item": trace}],
        },
    },
)

# Poll until the asynchronous service reaches a terminal state.
while True:
    agent_run = openai_client.evals.runs.retrieve(
        eval_id=agent_eval.id,
        run_id=agent_run.id,
    )
    if agent_run.status in ("completed", "failed", "canceled"):
        break
    print(f"Agent run status: {agent_run.status}")
    time.sleep(5)

# Print the new-portal link and retrieve the one row of detailed results.
print(f"Agent run finished: {agent_run.status}")
print("Open in new Foundry:", agent_run.report_url)
agent_output_items = list(
    openai_client.evals.runs.output_items.list(
        eval_id=agent_eval.id,
        run_id=agent_run.id,
    )
)
print(f"Scored rows: {len(agent_output_items)}")

## 6. Interpret the results

Review the three cloud runs together in **Microsoft Foundry -> Build -> Evaluations**:

- **Quality:** investigate failed labels or groundedness below the configured threshold. The bot may need a stricter prompt or better policy context.
- **Safety:** treat failed safety criteria as release blockers and continue with [`05-red-teaming.ipynb`](05-red-teaming.ipynb).
- **Agent behavior:** failed intent, adherence, or tool-call checks usually indicate instruction, tool-schema, or argument-extraction problems.

The portal should contain these evaluation definitions:

- `zava-support-bot-quality` over the versioned eight-row dataset.
- `zava-support-bot-safety` over the same dataset version.
- `zava-order-tracker-agent` over one inline structured trace.

Each code path prints a direct `report_url` for its exact run. The SDK `output_items` collections expose the same row-level scores, pass/fail labels, thresholds, and reasons programmatically.

> Rerunning a submission cell creates additional project assets and can incur model/evaluation charges. Reuse the printed evaluation and run IDs when you only need to inspect an existing result.

Next: [`04-tracing-and-monitoring.ipynb`](04-tracing-and-monitoring.ipynb) - capture live traces and set up continuous evaluation.